# 📈 Tutorial 3: Multi-Objective Pareto Optimization, TEA & ISO 14040 LCA

In this tutorial, you will:
1. Solve multi-objective trade-offs between Bio-Oil Yield, Biochar Carbon, and Economic Profit.
2. Rank Pareto-optimal solutions using **TOPSIS Multi-Criteria Decision Making (MCDM)**.
3. Perform 20-Year Discounted Cash Flow (DCF) techno-economic valuation (NPV, IRR, Payback).
4. Compute ISO 14040/14044 Scope 1-2-3 Life Cycle Assessment and permanent biochar carbon sequestration.

In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.optimization.pareto import ParetoOptimizer
from src.optimization.decision_maker import TOPSISDecisionMaker
from src.economics.run_economics import evaluate_plant_economics_and_lca

print("[*] Optimization and TEA/LCA modules loaded.")

## 1. Generating the Non-Dominated Pareto Frontier

In [ ]:
opt = ParetoOptimizer(feedstock_name="olive_pomace")
frontier = opt.generate_pareto_frontier(n_candidates=25)
non_dom = frontier.get_non_dominated_solutions()

print(f"Generated {len(non_dom)} Pareto non-dominated optimal solutions.\n")

ranked = TOPSISDecisionMaker.rank_solutions(frontier, profile_name="balanced_sustainability")
champion = ranked[0]
print(f"TOPSIS Champion Solution (Score: {champion['closeness_score']*100:.1f}%):")
for k, v in champion['setpoints'].items():
    print(f"  Setpoint {k:<20}: {v:.2f}")
for k, v in champion['objectives'].items():
    print(f"  Objective {k:<20}: {v:.2f}")

## 2. Techno-Economic Feasibility (20-Yr DCF) & ISO 14040/14044 Carbon Accounting

In [ ]:
econ_results = evaluate_plant_economics_and_lca(
    feedstock_name="olive_pomace",
    feed_rate_kg_h=100.0,
    reactor_temp_c=500.0,
    bio_oil_price_usd_kg=0.65,
    biochar_price_usd_kg=0.45,
    corc_price_usd_tonne=65.0
)

fin = econ_results["financial_viability_dcf"]
cap = econ_results["capital_expenditure_capex"]
op = econ_results["operational_expenditure_opex"]
lca = econ_results["life_cycle_assessment_lca"]

print(f"=== Financial Viability (20-Year DCF @ 10% WACC) ===")
print(f"  Total Capital Investment (TCI): ${cap['total_capital_investment_usd']:,.2f}")
print(f"  Annual Operating Cost (OPEX)  : ${op['total_opex_usd_yr']:,.2f}/yr")
print(f"  Net Present Value (NPV)       : ${fin['net_present_value_usd']:,.2f}")
print(f"  Internal Rate of Return (IRR) : {fin['internal_rate_of_return_pct']:.2f}%")
print(f"  Discounted Payback Period     : {fin['discounted_payback_years']:.1f} years")
print(f"  Levelized Cost of Bio-Oil     : ${fin['levelized_cost_bio_oil_usd_kg']:.3f}/kg\n")

print(f"=== ISO 14040/14044 Carbon Balance ===")
print(f"  Gross Scope 1+2+3 Emissions   : {lca['scope_emissions']['total_gross_emissions_co2e_kg_yr']:,.1f} kg CO2e/yr")
print(f"  Permanent Biochar Removal     : {lca['sequestration']['co2_sequestered_kg_yr']:,.1f} kg CO2/yr")
print(f"  Net Carbon Intensity          : {lca['carbon_intensity_g_co2e_per_mj_bio_oil']:.2f} g CO2eq/MJ (Carbon Negative: {lca['is_carbon_negative']})")